In [ ]:
import scanpy as sc
import numpy as npp
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import scipy.stats as stats
import gffutils

### Process gene expression data (GUIDE CAPTURE)

In [ ]:
gex = sc.read_h5ad("GEX2.h5ad")

# set the gene expression count matrix to be the sum of all three types of reads
gex.X = gex.layers["ambiguous"]+gex.layers["mature"]+gex.layers["nascent"]

In [ ]:
# count the number of gene expression UMIs per cell
counts = gex.X.sum(axis=1)
counts = np.asarray(counts).flatten()
gex.obs["GEX_UMIS"] = counts

In [ ]:
def plot_knee(umi_counts, ncells):
    # sort UMI counts in descending order
    sorted_counts = np.sort(umi_counts)[::-1]

    plt.figure(figsize=(5,5))
    plt.plot(list(range(len(sorted_counts))), sorted_counts, linewidth=4, color="black")

    plt.xlabel("Barcode rank")
    plt.ylabel("UMI counts")

    plt.yscale("log")
    plt.xscale("log")

    plt.axvline(ncells, linewidth=2, linestyle='--', color="red")
    
    print(f"UMIs at {ncells}th cell: {sorted_counts[ncells]}")

In [ ]:
plt.rcParams["font.size"] = 24
plt.rcParams["font.family"] = "Arial"
plt.rcParams["figure.figsize"] = (5,5)

plot_knee(gex.obs["GEX_UMIS"], ncells=6000)
plt.savefig("DC_GEX_knee.png", dpi=300, bbox_inches="tight")

In [ ]:
# only keep cells with >threshold UMIs
gex = gex[gex.obs["GEX_UMIS"]>1150]

In [ ]:
gex.obs["GEX_UMIS"].mean(), gex.obs["GEX_UMIS"].median()

### Process guide data (GUIDE CAPTURE)

In [ ]:
guide = sc.read_h5ad("GUIDE1.h5ad")
guide = guide.to_df()

In [ ]:
# merge the gene expression and guide objects
gex.obs = gex.obs.merge(guide, left_index=True, right_index=True, how="left")
# if a guide is not detected in a cell, set its count to 0
gex.obs = gex.obs.fillna(0)
# count the number of guide UMIs per cell
gex.obs["GUIDE_UMIS"] = gex.obs[guide.columns].sum(axis=1)

In [ ]:
# mixing plot - want points NOT to lie along the x or y axes
#sns.scatterplot(x=gex.obs["GEX_UMIS"], y=gex.obs["GUIDE_UMIS"], alpha=0.1)

### Process gene expression data (CROP)

In [ ]:
gex_crop = sc.read_h5ad("CROP_GEX1.h5ad")
gex_crop.X = gex_crop.layers["ambiguous"]+gex_crop.layers["mature"]+gex_crop.layers["nascent"]

In [ ]:
counts = gex_crop.X.sum(axis=1)
counts = np.asarray(counts).flatten()
gex_crop.obs["GEX_UMIS"] = counts

In [ ]:
plot_knee(gex_crop.obs["GEX_UMIS"], ncells=6200)
plt.savefig("CROP_GEX_knee.png", dpi=300, bbox_inches="tight")

In [ ]:
gex_crop = gex_crop[gex_crop.obs["GEX_UMIS"]>1180]

In [ ]:
gex_crop.obs["GEX_UMIS"].mean(), gex_crop.obs["GEX_UMIS"].median()

### Process guide data (CROP)

In [ ]:
guide_crop = sc.read_h5ad("CROP_GUIDE1.h5ad")
guide_crop = guide_crop.to_df()

In [ ]:
gex_crop.obs = gex_crop.obs.merge(guide_crop, left_index=True, right_index=True, how="left")
gex_crop.obs = gex_crop.obs.fillna(0)
gex_crop.obs["GUIDE_UMIS"] = gex_crop.obs[guide_crop.columns].sum(axis=1)

In [ ]:
#sns.scatterplot(x=gex_crop.obs["GEX_UMIS"], y=gex_crop.obs["GUIDE_UMIS"])

### Histogram of number of gene expression UMIs for both libraries

In [ ]:
counts = pd.DataFrame({"Direct capture": gex.obs["GEX_UMIS"],
                       "CROP-seq": gex_crop.obs["GEX_UMIS"]})
# convert the table from a wide to a long table
counts = counts.melt().dropna()
counts.columns = ["Method", "# GEX UMIs"]

In [ ]:
sns.violinplot(x="# GEX UMIs", y="Method", 
               data=counts, log_scale=True,
               palette=["#ffd479","#76d6ff"])
plt.xlabel("# of gene expression UMIs")
plt.ylabel("")
plt.savefig("GEX_UMIS.png", dpi=300, bbox_inches="tight")

### Histogram of number of guide UMIs per guide for both libraries

In [ ]:
guide_counts = pd.DataFrame({"Direct capture": gex.obs["GUIDE_UMIS"],
                             "CROP-seq": gex_crop.obs["GUIDE_UMIS"]})
guide_counts = guide_counts.melt().dropna()
guide_counts.columns = ["Method", "# GUIDE UMIs"]

In [ ]:
sns.violinplot(x="# GUIDE UMIs", y="Method", 
               data=guide_counts[guide_counts["# GUIDE UMIs"]>0], 
               log_scale=True, palette=["#ffd479","#76d6ff"])
plt.xlabel("# of guide UMIs")
plt.ylabel("")
plt.savefig("GUIDE_UMIS.png", dpi=300, bbox_inches="tight")

In [ ]:
# calculate fraction of cells with nonzero 
guide_frac_nonzero = np.mean(gex.obs["GUIDE_UMIS"]>0)
crop_guide_frac_nonzero = np.mean(gex_crop.obs["GUIDE_UMIS"]>0)

In [ ]:
sns.barplot(y=["direct", "crop"],
            x=[guide_frac_nonzero, crop_guide_frac_nonzero])
# TODO: place a number on the barplot

In [ ]:
((gex.obs.iloc[:,1:]>=5).sum(axis=1)>0).mean()

In [ ]:
((gex_crop.obs.iloc[:,1:]>=5).sum(axis=1)>0).mean()

### Distributions of counts for individual guides (GUIDE)

In [ ]:
guides = guide.columns.tolist()

In [ ]:
cutoff = 5

for g in ["TP53_+_7590780.23-P1P2"]:
    gcounts = gex.obs[g]
    gcounts = gcounts[gcounts>0]

    gcounts = pd.DataFrame(gcounts)
    gcounts["real"] = False
    gcounts.loc[gcounts[g]>cutoff,"real"] = True

    if gcounts["real"].sum()<100:
        continue
    
    sns.histplot(x=g, hue="real", data=gcounts, log_scale=True, 
                common_norm=False, stat="percent", alpha=0.5,
                palette=["#a9a9a9","#ffd479"])
    
    plt.axvline(cutoff, linestyle='--', color="red")
    plt.legend("", frameon=False)
    plt.savefig("DC_example_guide.png", dpi=300, bbox_inches="tight")
    plt.show()

### Distributions of counts for individual guides (CROP)

In [ ]:
cutoff = 3

for g in ["TP53_+_7590780.23-P1P2"]:
    gcounts = gex_crop.obs[g]
    gcounts = gcounts[gcounts>0]

    gcounts = pd.DataFrame(gcounts)
    gcounts["real"] = False
    gcounts.loc[gcounts[g]>cutoff,"real"] = True

    if gcounts["real"].sum()<100:
        continue
    
    sns.histplot(x=g, hue="real", data=gcounts, log_scale=True, 
                 common_norm=False, stat="percent", alpha=0.5,
                 palette=["#a9a9a9","#76d6ff"])
    
    plt.axvline(cutoff, linestyle='--', color="red")
    plt.legend("", frameon=False)
    plt.savefig("CROP_example_guide.png",  dpi=300, bbox_inches="tight")
    plt.show()

### Make saturation curves for guide libraries

In [ ]:
results = []

for counts in [1000000,3000000,10000000,30000000]:
    adata = sc.read_h5ad(f"GUIDE_S1.{counts}_kb_kite/counts_unfiltered_modified/adata.h5ad")
    cells = list(set(gex.obs.index) & set(adata.obs.index))
    adata = adata[cells]
    med_umis = np.median(np.asarray(adata.X.sum(axis=1)).flatten())
    nreads = counts/len(cells)
    results.append([nreads,med_umis,"Direct capture"])

for counts in [1000000,3000000,10000000,30000000]:
    adata = sc.read_h5ad(f"CROP_GUIDE1_S4.{counts}_kb_kite/counts_unfiltered_modified/adata.h5ad")
    cells = list(set(gex_crop.obs.index) & set(adata.obs.index))
    adata = adata[cells]
    med_umis = np.median(np.asarray(adata.X.sum(axis=1)).flatten())
    nreads = counts/len(cells)
    results.append([nreads,med_umis,"CROP-seq"])

In [ ]:
results = pd.DataFrame(results)
results.columns = ["Reads per cell","UMIs per cell","Method"]

In [ ]:
sns.lineplot(x="Reads per cell", y="UMIs per cell", hue="Method", data=results,
             palette=["#ffd479","#76d6ff"], linewidth=4, marker='o', markersize=15)
plt.xlabel("Number of reads per cell")
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))

plt.savefig("guide_saturation_curve.png",  dpi=300, bbox_inches="tight")

### Make UMAP (guide)

In [ ]:
db = gffutils.FeatureDB("/Users/tkzeng/Dropbox/engreitz/statin_pilot/gencode.v46.annotation.db")

id_to_name = {}

for gene in db.features_of_type("gene"):
    gene_id = gene["gene_id"][0]
    gene_name = gene["gene_name"][0]
    id_to_name[gene_id] = gene_name

In [ ]:
gex.var.index = gex.var.index.map(id_to_name)
gex.var_names_make_unique()

gex.var["mt"] = gex.var_names.str.startswith("MT-")

sc.pp.calculate_qc_metrics(
    gex, qc_vars=["mt"], inplace=True, log1p=True
)

# sc.pl.violin(
#     gex,
#     ["GEX_UMIS", "pct_counts_mt"],
#     jitter=0.4,
#     multi_panel=True,
# )

sc.pp.filter_cells(gex, min_genes=100)
sc.pp.filter_genes(gex, min_cells=3)

# Saving count data
gex.layers["counts"] = gex.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(gex)
# Logarithmize the data
sc.pp.log1p(gex)

sc.pp.highly_variable_genes(gex, n_top_genes=2000)

sc.tl.pca(gex)
sc.pp.neighbors(gex)
sc.tl.umap(gex)

In [ ]:
# sc.pl.umap(gex)

### Calculate knockdown (guide)

In [ ]:
gex_copy = gex.copy()
    
guide_counts = gex_copy.obs[guides]
guide_assign = guide_counts[guide_counts>=5]
genes = [x.split('_')[0] for x in guides]
guide_assign.columns = genes

gene_assign = {}
for gene in set(genes):
    gene_assign[gene] = guide_assign[gene].sum(axis=1)>0
gene_assign = pd.DataFrame(gene_assign)
gex_copy.obs = gene_assign

In [ ]:
all_de = []
for gene in set(genes):
    gex_copy.obs["targeted"] = gex_copy.obs[gene].astype(str)
    sc.tl.rank_genes_groups(gex_copy, groupby="targeted", method="t-test_overestim_var")
    de = sc.get.rank_genes_groups_df(gex_copy, group="True")
    de = de[de["names"]==gene]
    de["ncells"] = np.sum(gex_copy.obs["targeted"]=="True")
    all_de.append(de)
all_de = pd.concat(all_de)

In [ ]:
#sns.histplot(all_de["logfoldchanges"][all_de["logfoldchanges"]>-15])

### Make UMAP (crop)

In [ ]:
gex_crop.var.index = gex_crop.var.index.map(id_to_name)
gex_crop.var_names_make_unique()

gex_crop.var["mt"] = gex_crop.var_names.str.startswith("MT-")

sc.pp.calculate_qc_metrics(
    gex_crop, qc_vars=["mt"], inplace=True, log1p=True
)

# sc.pl.violin(
#     gex_crop,
#     ["GEX_UMIS", "pct_counts_mt"],
#     jitter=0.4,
#     multi_panel=True,
# )

sc.pp.filter_cells(gex_crop, min_genes=100)
sc.pp.filter_genes(gex_crop, min_cells=3)

# Saving count data
gex_crop.layers["counts"] = gex_crop.X.copy()

# Normalizing to median total counts
sc.pp.normalize_total(gex_crop)
# Logarithmize the data
sc.pp.log1p(gex_crop)

sc.pp.highly_variable_genes(gex_crop, n_top_genes=2000)

sc.tl.pca(gex_crop)
sc.pp.neighbors(gex_crop)
sc.tl.umap(gex_crop)

### Calculate knockdown (crop)

In [ ]:
gex_crop_copy = gex_crop.copy()

guide_counts = gex_crop_copy.obs[guides]
guide_assign = guide_counts[guide_counts>=3]
genes = [x.split('_')[0] for x in guides]
guide_assign.columns = genes

gene_assign = {}
for gene in set(genes):
    gene_assign[gene] = guide_assign[gene].sum(axis=1)>0
gene_assign = pd.DataFrame(gene_assign)
gex_crop_copy.obs = gene_assign

In [ ]:
all_de_crop = []
for gene in set(genes):
    gex_crop_copy.obs["targeted"] = gex_crop_copy.obs[gene].astype(str)
    sc.tl.rank_genes_groups(gex_crop_copy, groupby="targeted", method="t-test_overestim_var")
    de = sc.get.rank_genes_groups_df(gex_crop_copy, group="True")
    de = de[de["names"]==gene]
    de["ncells"] = np.sum(gex_crop_copy.obs["targeted"]=="True")
    all_de_crop.append(de)
all_de_crop = pd.concat(all_de_crop)

In [ ]:
all_de["method"] = "Direct capture"
all_de_crop["method"] = "CROP-seq"
de_direct_crop = pd.concat([all_de,all_de_crop])

In [ ]:
de_direct_crop = de_direct_crop[de_direct_crop["names"]!="JUNB"]
de_direct_crop.loc[de_direct_crop["logfoldchanges"]>0,["pvals","pvals_adj"]] = 1

In [ ]:
de_direct_crop[de_direct_crop["method"]=="Direct capture"].sort_values(by="pvals")

In [ ]:
de_direct_crop[de_direct_crop["method"]=="CROP-seq"].sort_values(by="pvals")

In [ ]:
p_values_1 = np.sort(de_direct_crop[de_direct_crop["method"]=="Direct capture"]["pvals"])
p_values_2 = np.sort(de_direct_crop[de_direct_crop["method"]=="CROP-seq"]["pvals"])

# Sort the p-values
sorted_p_values_1 = np.sort(p_values_1)
sorted_p_values_2 = np.sort(p_values_2)

# Calculate the theoretical quantiles (expected under the null hypothesis)
theoretical_quantiles = np.linspace(0, 1, max(len(sorted_p_values_1), len(sorted_p_values_2)))

# Transform to uniform distribution
expected_p_values = -np.log10(theoretical_quantiles[1:])  # Avoid log(0)
observed_p_values_1 = -np.log10(sorted_p_values_1[1:])  # Avoid log(0)
observed_p_values_2 = -np.log10(sorted_p_values_2[1:])  # Avoid log(0)

# Create the QQ plot
plt.scatter(expected_p_values, observed_p_values_1, edgecolor='k', linewidth=0.5, facecolor="#ffd479", label="Direct capture", s=100)
plt.scatter(expected_p_values, observed_p_values_2, edgecolor='k', linewidth=0.5, facecolor="#76d6ff", label="CROP-seq", s=100)
plt.plot([min(expected_p_values), max(expected_p_values)],
         [min(expected_p_values), max(expected_p_values)], color="gray", ls='--')
plt.xlabel('Expected -log10(p)')
plt.ylabel('Observed -log10(p)')
plt.title('Target knockdown')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.savefig("knockdown_QQ_plot.png",  dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
sns.pointplot(y="method", x="logfoldchanges", data=de_direct_crop, join=False)

In [ ]:
"TP53" in de_direct_crop["names"].tolist()

In [ ]:
sns.pointplot(y="method", x="ncells", data=de_direct_crop, join=False)

In [ ]:
example_crop = gex_crop_copy.obs.merge(gex_crop_copy[:,"ITGB1"].to_df(),
                                    left_index=True, right_index=True, suffixes=["_assign","_expr"])

sns.violinplot(y="ITGB1_expr",
               x="ITGB1_assign",
               data=example_crop,
               color="#76d6ff")
plt.xlabel("ITGB1 guide")
plt.ylabel("ITGB1 expression")
plt.savefig("example.crop.png", dpi=300, bbox_inches="tight")

In [ ]:
example_dc = gex_copy.obs.merge(gex_copy[:,"ITGB1"].to_df(),
                                left_index=True, right_index=True, suffixes=["_assign","_expr"])

sns.violinplot(y="ITGB1_expr",
               x="ITGB1_assign",
               data=example_dc,
               color="#ffd479")
plt.xlabel("ITGB1 guide")
plt.ylabel("ITGB1 expression")
plt.savefig("example.dc.png", dpi=300, bbox_inches="tight")

In [ ]:
pd.concat([gex_crop_copy.obs,
           gex_crop_copy[:,"TP53"].to_df()], axis=1)

In [ ]:
gex_crop_copy[:,"TP53"]

In [ ]:
(gex_crop.obs[guides]>=3).sum(axis=1).mean()

In [ ]:
sns.histplot((gex.obs[guides]>=5).sum(axis=1).tolist())

In [ ]:
(gex.obs[guides]>=5).sum(axis=1)